# Notebook 03 — Derivative-Constrained Deep Smoothing of the Implied Volatility Surface

**Goal.** NB02 quantified the central tension of parametric smoothing: per-slice **SVI fits best
but can violate butterfly no-arbitrage**; **SSVI is arbitrage-free by construction but less
accurate**. This notebook asks the thesis question: *can a neural network fit as well as SVI while
keeping (almost) the no-arbitrage guarantees of SSVI?*

**Method — a synthesis of the two reference papers:**

- **Ackerer, Tagasovska & Vatter (NeurIPS 2020), *Deep Smoothing of the IVS*** — total variance as
  the **product of an arbitrage-free prior and a neural corrector**:
  $$ w_\theta(k,\tau) = \underbrace{w_{\text{SSVI}}(k,\tau)}_{\text{prior (NB02)}} \times \underbrace{\mathcal C_\theta(k,\tau)}_{\text{neural corrector }>0}. $$

- **Hoshisashi, Phelan & Barucca (2024), *No-Arbitrage Deep Calibration (DCNN)*** — the network's
  **exact derivatives by automatic differentiation**, soft no-arbitrage penalties on dense
  collocation grids distinct from the quotes.

---

## Version history of the methodology (kept on purpose — each step is a thesis paragraph)

- **v3** reported `pen_cal = 0.000e+00` and a 28% calendar-violation rate *simultaneously*. Both
  numbers were true: the exponential τ-collocation left a 0.41y hole between its last two nodes,
  and the surface violated *between* the nodes. Soft constraints are enforced only where sampled
  (Chataigner's caveat, literally).
- **v5** fixed it with **constraint-specific collocation grids**: butterfly and calendar have
  *opposite* grid needs, so they no longer share one. The calendar penalty then genuinely
  arbitrates (38.1% → 1.1% violations on the stress test). v5 also removed a data leak in the
  sparsity study (the prior was calibrated on the full quote set inside the subsampling loop) —
  which exposed the next weakest link:
- **v6 (this file)** fixes the two remaining defects v5 revealed:
  1. **Prior-calibration instability under sparsity.** With the leak removed, the 50% subsample
     produced a 14.2 vp RMSE — an explosion, not a degradation. Root cause: `fit_prior` estimated
     per-slice ATM variance with `onp.interp(0.0, ...)`, which **clamps** outside the quoted
     k-range; a subsampled slice with all quotes on one side of the money silently returned a wing
     value, corrupting θ(τ) and hence (α, β). A *wrong* prior is worse than a *flat* one (14.2 vp
     vs 1.7 vp for `no_prior`): the multiplicative corrector must first undo it. Fixed with a
     robust nearest-ATM proxy + joint 5-parameter refinement, plus **prior-only RMSE
     instrumentation** and **multi-seed subsampling**.
  2. **Adam-restart transient.** `train_logged` re-initialised the Adam state at every logging
     chunk; with 800 epochs / 12 logs the final chunk was 8 epochs long, so the *returned*
     parameters sat mid-transient (the visible end-of-curve spike in the dynamics plot). The Adam
     state now persists across chunks.

**Why activations must be $C^2$.** The loss involves $w_{kk}$; ReLU has zero second derivative
almost everywhere and kills the butterfly penalty. We use **tanh** ($C^\infty$).

**Autodiff engine.** `autograd` (HIPS): exact nested derivatives of a NumPy MLP via
`elementwise_grad`, validated in **both** $k$ (against closed-form SVI) and $\tau$ (against
central differences on the SSVI prior). The τ check is not decorative: the calendar penalty
depends entirely on that path.



## 0. Imports & Configuration



In [55]:
import os, time, zlib
from pathlib import Path

import autograd.numpy as np
from autograd import elementwise_grad as egrad, grad
import numpy as onp
import polars as plr
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import minimize



In [56]:
# --- config ---
REAL_PARQUET  = Path(os.environ.get("THESIS_OPT_PARQUET", "data/clean/option_prices_clean.parquet"))
OUT_DIR       = Path(os.environ.get("THESIS_OUT_DIR", "data/clean")); OUT_DIR.mkdir(parents=True, exist_ok=True)
SEED          = 0
HIDDEN        = (64, 64)      # corrector MLP width
EPOCHS_MAIN   = 1500          # final synthetic model (display only)
EPOCHS_ABL    = 800           # ablations: SAME budget for every variant (fair comparison)
EPOCHS_REAL   = 1500          # per real day
EPOCHS_STRESS = 3000          # stress tests: the models must actually FIT the distortion
LAMBDA_BFLY   = 10.0
LAMBDA_CAL    = 10.0
LAMBDA_WING   = 0.1           # light Ackerer C6-style linearity penalty on far wings

# --- constraint-specific collocation (v5 design) ---
# Butterfly and calendar have OPPOSITE grid requirements, so they get separate grids:
#   butterfly g>=0 : bites at SHORT maturities (w small => the slope term w'^2/4w blows up)
#                    and needs k-density at the money  -> cube-root k, HYBRID (exp U uni) tau
#   calendar  wt>=0: is a TAU-derivative constraint that can bite anywhere -> what matters is
#                    a BOUNDED NODE GAP in tau; d_tau w costs one first derivative, so a dense
#                    uniform tau grid is affordable
COLLOC_NK     = 24            # k-resolution of the butterfly grid
COLLOC_NT     = 14            # tau-resolution PER FAMILY of the butterfly hybrid grid (exp U uni)
CAL_NK        = 18            # k-resolution of the calendar grid (coarser: wt varies slowly in k)
CAL_NT        = 80            # tau-resolution of the calendar grid (uniform, dense)

EXT_FACTOR    = 2.0           # arbitrage audited on EXT_FACTOR x the quoted k-range ...
K_EXT_CAP     = 1.5           # ... but never beyond |k| = K_EXT_CAP, and never INSIDE the quotes
LIMIT_DATES   = 2             # real-data quick pass; None = all days
MIN_PTS_SLICE = 6
HOLDOUT_FRAC  = 0.20
SPARSITY_SEEDS = (7, 17, 27)  # subsampling seeds for the sparsity study (one seed = one lottery)
RUN_LAM0_CHECK = False        # True: re-run lambda=0 on the last real day with 3x epochs, 3 seeds

print(f"HIDDEN={HIDDEN} | epochs main/abl/stress/real = "
      f"{EPOCHS_MAIN}/{EPOCHS_ABL}/{EPOCHS_STRESS}/{EPOCHS_REAL}")
print(f"bfly colloc {COLLOC_NK} x (exp{COLLOC_NT} U uni{COLLOC_NT}) | "
      f"calendar colloc {CAL_NK} x uni{CAL_NT} | "
      f"lambdas bfly/cal/wing = {LAMBDA_BFLY}/{LAMBDA_CAL}/{LAMBDA_WING} | "
      f"ext factor {EXT_FACTOR}, cap |k|<={K_EXT_CAP}")

rng = onp.random.default_rng(SEED)


def stable_seed(d):
    """Deterministic per-date seed (NB02 protocol); Python's hash() is salted per process."""
    return zlib.crc32(str(d).encode("utf-8"))



HIDDEN=(64, 64) | epochs main/abl/stress/real = 1500/800/3000/1500
bfly colloc 24 x (exp14 U uni14) | calendar colloc 18 x uni80 | lambdas bfly/cal/wing = 10.0/10.0/0.1 | ext factor 2.0, cap |k|<=1.5


## 1. Building blocks and a hard validation of the autodiff machinery

Everything lives in **total-variance space** $w(k,\tau)=\sigma_{\text{IV}}^2\tau$. Before trusting
autodiff-of-a-network we validate it against ground truth in **both** input directions:

- **in $k$**: against the closed-form first and second derivatives of raw SVI;
- **in $\tau$**: against central finite differences on the SSVI prior.

An untested derivative in a loss term is an untested loss term.



In [57]:
# ---------- SSVI prior (differentiable, power-law theta) ----------
def ssvi_w_np(k, theta, rho, eta, gamma):
    phi = eta * theta ** (-gamma)
    return 0.5 * theta * (1 + rho * phi * k + np.sqrt((phi * k + rho) ** 2 + (1 - rho ** 2)))


def make_prior(rho, eta, gamma, alpha, beta):
    """SSVI prior with smooth increasing ATM total variance theta(tau) = alpha * tau**beta.
    Differentiable in (k, tau) -> usable inside the penalties."""
    def prior(k, tau):
        theta = alpha * tau ** beta
        return ssvi_w_np(k, theta, rho, eta, gamma)
    return prior


# ---------- raw SVI closed forms for the AD validation ----------
def svi_raw(k, a, b, rho, m, s):  return a + b * (rho * (k - m) + np.sqrt((k - m) ** 2 + s ** 2))
def svi_p(k, a, b, rho, m, s):    return b * (rho + (k - m) / np.sqrt((k - m) ** 2 + s ** 2))
def svi_pp(k, a, b, rho, m, s):   return b * s ** 2 / ((k - m) ** 2 + s ** 2) ** 1.5


def durrleman_g(k, w, wk, wkk):
    return (1 - k * wk / (2 * w)) ** 2 - (wk ** 2 / 4) * (1 / w + 0.25) + wkk / 2


# --- validation A: egrad in k, vs closed-form SVI derivatives ---
P = dict(a=0.02, b=0.15, rho=-0.4, m=0.05, s=0.25)
f  = lambda k: svi_raw(k, **P)
kk = onp.linspace(-0.8, 0.8, 201)
e1 = onp.max(onp.abs(egrad(f)(kk) - svi_p(kk, **P)))
e2 = onp.max(onp.abs(egrad(egrad(f))(kk) - svi_pp(kk, **P)))
print(f"AD vs closed form (in k):     |w' err|  = {e1:.2e}   |w'' err| = {e2:.2e}")
assert e1 < 1e-8 and e2 < 1e-8, "autodiff machinery in k is broken"

# --- validation B: egrad in TAU, vs central differences on the prior ---
_pr_v = make_prior(rho=-0.5, eta=0.9, gamma=0.4, alpha=0.045, beta=0.95)
_tt = onp.linspace(0.05, 1.5, 101)
_k0 = onp.full_like(_tt, -0.1)
_d_ad = egrad(lambda t: _pr_v(_k0, t))(_tt)
_d_fd = (_pr_v(_k0, _tt + 1e-6) - _pr_v(_k0, _tt - 1e-6)) / 2e-6
e3 = onp.max(onp.abs(_d_ad - _d_fd))
print(f"AD vs finite diff (in tau):   |d_tau w err| = {e3:.2e}")
assert e3 < 1e-6, "the tau-derivative path (and hence the calendar penalty) is broken"

print("AD machinery validated in BOTH k and tau.")



AD vs closed form (in k):     |w' err|  = 5.55e-17   |w'' err| = 1.11e-16
AD vs finite diff (in tau):   |d_tau w err| = 1.43e-11
AD machinery validated in BOTH k and tau.


## 2. Architecture, constraint-specific collocation and the penalized loss

**Corrector.** MLP $(k,\tau)\mapsto\mathbb R$, tanh activations; $\mathcal C_\theta=\exp(\text{MLP})$,
positive and $\approx 1$ at initialization — training *starts at the prior*. Inputs rescaled to
$[-1,1]^2$.

### Collocation: one grid per constraint

| constraint | where it bites | grid |
|---|---|---|
| butterfly $g \ge 0$ | short maturities, near the money | cube-root $k$ (dense ATM), **hybrid** τ (exp ∪ uniform) |
| calendar $\partial_\tau w \ge 0$ | anywhere; it is a τ-derivative | coarse $k$, **uniform-dense** τ (`CAL_NT` nodes) |

The calendar penalty needs only one first derivative per node, so density in τ is cheap; the
butterfly penalty needs $w, w_k, w_{kk}$, so it keeps a compact hybrid grid. Both grids span the
**capped extended range** `ext_range` — the constraints are enforced (not just audited) beyond
the quotes.

### The audit grid stays independent

The audit grid (`surf_g_and_cal`) is uniform, finer, and **not** a subset of either collocation
grid. That independence is what exposed the v3 blind spot. Every penalty is therefore reported
twice — at its nodes (`pen_*`) and on the audit grid (`pen_*_ext`) — and the gap between the two
is the honest measure of what collocation cannot see (`blind_*` flags).



In [58]:
# ---------- MLP with autograd-friendly params ----------
def init_mlp(sizes, seed=0, scale=0.1):
    r = onp.random.default_rng(seed)
    return [(np.array(r.normal(0, scale, (m, n))), np.zeros(n))
            for m, n in zip(sizes[:-1], sizes[1:])]


def mlp_forward(params, X):
    h = X
    for W, b in params[:-1]:
        h = np.tanh(h @ W + b)
    W, b = params[-1]
    return (h @ W + b)[:, 0]


def make_model(prior, k_scale, t_mid, t_scale):
    """w(params, k, tau) = prior(k, tau) * exp(MLP(k~, tau~))."""
    def w_model(params, k, tau):
        X = np.stack([k / k_scale, (tau - t_mid) / t_scale], axis=1)
        return prior(k, tau) * np.exp(mlp_forward(params, X))
    return w_model


def relu(x):
    return np.maximum(x, 0.0)


def iv_target_weights(w, tau):
    """Same delta-method weights as NB02: least squares in w ~ least squares in IV."""
    wt = 1.0 / (4.0 * onp.maximum(onp.asarray(w, float), 1e-10) * onp.asarray(tau, float))
    return wt / wt.mean()


def ext_range(k_lo, k_hi, ext=EXT_FACTOR, cap=K_EXT_CAP):
    """Extended k-range for penalties AND audit: EXT_FACTOR x the quoted range, clipped to
    +/- K_EXT_CAP, but NEVER shrunk inside the quoted range. On real SPX, deep-OTM put quotes
    reach k ~ -2.2; 2x that is k = -4.4, a strike at ~1% of spot -- auditing there inflates
    violation rates with no economic content."""
    lo = max(ext * k_lo, -cap)
    hi = min(ext * k_hi, cap)
    return float(min(lo, k_lo)), float(max(hi, k_hi))


def tau_grid_hybrid(t_lo, t_hi, nt):
    """Exponential (dense short end) UNION uniform (bounded node gap). Returns nodes + max gap."""
    t_lo = max(float(t_lo), 1.0 / 365.0)
    t_hi = float(t_hi)
    t_exp = onp.exp(onp.linspace(onp.log(t_lo), onp.log(t_hi), nt))
    t_uni = onp.linspace(t_lo, t_hi, nt)
    tg = onp.unique(onp.round(onp.concatenate([t_exp, t_uni]), 8))
    max_gap = float(onp.max(onp.diff(tg))) if len(tg) > 1 else 0.0
    return tg, max_gap


def make_collocation(k_lo, k_hi, t_lo, t_hi, verbose=False):
    """Constraint-specific grids spanning the capped extended k-range. Returns a dict:
       kc, tc         -> butterfly grid (cube-root k, hybrid tau)
       kcal, tcal     -> calendar grid  (uniform k, uniform-dense tau)
       kw, tw         -> far-wing points for the C6 linearity penalty
       gap_bfly, gap_cal -> realised max tau node gaps (the blind-spot budget)"""
    k_lo = min(float(k_lo), -0.05)
    k_hi = max(float(k_hi), 0.05)
    e_lo, e_hi = ext_range(k_lo, k_hi)

    # butterfly grid
    xk = onp.linspace(-((-e_lo) ** (1 / 3)), e_hi ** (1 / 3), COLLOC_NK)
    kg = xk ** 3
    tg_b, gap_b = tau_grid_hybrid(t_lo, t_hi, COLLOC_NT)
    Kc, Tc = onp.meshgrid(kg, tg_b)

    # calendar grid
    kg_c = onp.linspace(e_lo, e_hi, CAL_NK)
    tg_c = onp.linspace(max(float(t_lo), 1.0 / 365.0), float(t_hi), CAL_NT)
    gap_c = float(onp.max(onp.diff(tg_c)))
    Kl, Tl = onp.meshgrid(kg_c, tg_c)

    # far-wing points for the C6 linearity penalty, also capped
    w_lo = max(3 * k_lo, -K_EXT_CAP * 1.5)
    w_hi = min(3 * k_hi, K_EXT_CAP * 1.5)
    kw_pts = onp.unique(onp.array([2 * k_lo, w_lo, 2 * k_hi, w_hi]))
    Kw, Tw = onp.meshgrid(kw_pts, tg_b)

    if verbose:
        print(f"  bfly colloc: {Kc.size} pts (k in [{kg.min():+.3f},{kg.max():+.3f}], "
              f"{len(tg_b)} tau nodes, max gap {gap_b:.3f}y) | "
              f"calendar colloc: {Kl.size} pts ({CAL_NT} tau nodes, max gap {gap_c:.3f}y) | "
              f"wings {Kw.size} pts")
    return dict(kc=Kc.ravel(), tc=Tc.ravel(), kcal=Kl.ravel(), tcal=Tl.ravel(),
                kw=Kw.ravel(), tw=Tw.ravel(), gap_bfly=gap_b, gap_cal=gap_c)


def make_loss(w_model, kq, tq, wq, wtq, G, lam_b, lam_c, lam_w=LAMBDA_WING):
    """kq,tq,wq,wtq: sparse quotes + IV-target weights. G: collocation dict (see above)."""
    kc, tc, kcal, tcal, kw, tw = G["kc"], G["tc"], G["kcal"], G["tcal"], G["kw"], G["tw"]

    def loss(params):
        fit  = np.mean(wtq * (w_model(params, kq, tq) - wq) ** 2)
        # butterfly on ITS grid (needs w, w_k, w_kk)
        wk   = egrad(lambda k: w_model(params, k, tc))(kc)
        wkk  = egrad(egrad(lambda k: w_model(params, k, tc)))(kc)
        wc   = w_model(params, kc, tc)
        gval = durrleman_g(kc, wc, wk, wkk)
        pen_b = np.mean(relu(-gval) ** 2)
        # calendar on ITS grid (one first derivative, dense in tau)
        wt_  = egrad(lambda t: w_model(params, kcal, t))(tcal)
        pen_c = np.mean(relu(-wt_) ** 2)
        # positivity + far-wing linearity
        pen_f = np.mean(relu(-wc) ** 2)
        wkk_w = egrad(egrad(lambda k: w_model(params, k, tw)))(kw)
        pen_w = np.mean(wkk_w ** 2)
        return fit + lam_b * pen_b + lam_c * pen_c + lam_w * pen_w + 100.0 * pen_f
    return loss


# ---------- Adam with PERSISTENT state ----------
# v5 re-initialised (m, v, t) at every logging chunk. With 800 epochs and 12 logs the final chunk
# was 8 epochs: the returned parameters sat mid-transient of an optimizer restart (the visible
# end-of-curve spike in the dynamics plot), so every reported metric was measured at a perturbed
# point. State now threads through chunks.
def adam_init(params):
    return dict(m=[(onp.zeros_like(W), onp.zeros_like(b)) for W, b in params],
                v=[(onp.zeros_like(W), onp.zeros_like(b)) for W, b in params], t=0)


def adam(loss, params, epochs, lr=5e-3, state=None):
    g = grad(loss)
    st = state if state is not None else adam_init(params)
    m, v, t = st["m"], st["v"], st["t"]
    b1, b2, eps = 0.9, 0.999, 1e-8
    for _ in range(epochs):
        t += 1
        gr = g(params)
        new = []
        for i, ((W, b), (gW, gb)) in enumerate(zip(params, gr)):
            mW, mb = m[i]; vW, vb = v[i]
            mW = b1 * mW + (1 - b1) * gW; mb = b1 * mb + (1 - b1) * gb
            vW = b2 * vW + (1 - b2) * gW ** 2; vb = b2 * vb + (1 - b2) * gb ** 2
            m[i] = (mW, mb); v[i] = (vW, vb)
            mWh, mbh = mW / (1 - b1 ** t), mb / (1 - b1 ** t)
            vWh, vbh = vW / (1 - b2 ** t), vb / (1 - b2 ** t)
            new.append((W - lr * mWh / (onp.sqrt(vWh) + eps),
                        b - lr * mbh / (onp.sqrt(vbh) + eps)))
        params = new
    return params, dict(m=m, v=v, t=t)



In [59]:
def surf_g_and_cal(w_model, params, k0, k1, t0, t1, nk=60, nt=25):
    """g(k,tau) and d_tau w on an arbitrary domain (autodiff, vectorized).
    UNIFORM in both directions and deliberately NOT a subset of any collocation grid:
    its independence is load-bearing (it exposed the v3 blind spot)."""
    kg = onp.linspace(k0, k1, nk); tg = onp.linspace(t0, t1, nt)
    Kf, Tf = onp.meshgrid(kg, tg); kf, tf = Kf.ravel(), Tf.ravel()
    wk  = egrad(lambda k: w_model(params, k, tf))(kf)
    wkk = egrad(egrad(lambda k: w_model(params, k, tf)))(kf)
    wt_ = egrad(lambda t: w_model(params, kf, t))(tf)
    wv  = w_model(params, kf, tf)
    G  = onp.asarray(durrleman_g(kf, wv, wk, wkk)).reshape(nt, nk)
    WT = onp.asarray(wt_).reshape(nt, nk)
    return kg, tg, G, WT


def viol_overlay(Z, x, y):
    """Binary red mask where Z < 0 -- ANY negative value, regardless of magnitude (honest but
    blunt: read it together with the printed min values)."""
    M = onp.where(onp.asarray(Z) < 0, 1.0, onp.nan)
    return go.Heatmap(z=M, x=x, y=y, showscale=False,
                      colorscale=[[0, "rgba(214,39,40,0.9)"], [1, "rgba(214,39,40,0.9)"]])


BLIND_TOL = 1e-10   # a penalty below this at the nodes counts as "satisfied at the nodes"


def loss_components(w_model, params, kq, tq, wq, wtq, G, ext_dom):
    """Fit + penalties measured on BOTH grids (nodes vs independent audit).
    blind_* flags fire when a penalty is ~0 at its nodes while the audit grid still sees
    violations: a collocation-resolution failure, not a model failure (the v3 pathology)."""
    kc, tc, kcal, tcal = G["kc"], G["tc"], G["kcal"], G["tcal"]
    fit  = float(np.mean(wtq * (w_model(params, kq, tq) - wq) ** 2))

    wk   = egrad(lambda k: w_model(params, k, tc))(kc)
    wkk  = egrad(egrad(lambda k: w_model(params, k, tc)))(kc)
    wc   = w_model(params, kc, tc)
    gval = durrleman_g(kc, wc, wk, wkk)
    pen_bfly = float(np.mean(relu(-gval) ** 2))

    wt_  = egrad(lambda t: w_model(params, kcal, t))(tcal)
    pen_cal = float(np.mean(relu(-wt_) ** 2))

    _, _, Ge, WTe = surf_g_and_cal(w_model, params, *ext_dom)
    pen_bfly_ext = float(onp.mean(onp.maximum(-Ge, 0.0) ** 2))
    pen_cal_ext  = float(onp.mean(onp.maximum(-WTe, 0.0) ** 2))
    bfly_viol = float(100 * onp.mean(Ge < -1e-8))
    cal_viol  = float(100 * onp.mean(WTe < -1e-8))

    return dict(
        fit=fit,
        pen_bfly=pen_bfly, pen_cal=pen_cal,
        pen_bfly_ext=pen_bfly_ext, pen_cal_ext=pen_cal_ext,
        min_g=float(np.min(gval)),
        min_g_ext=float(Ge.min()),
        min_cal_nodes=float(np.min(wt_)),
        min_cal_ext=float(WTe.min()),
        bfly_viol_pct_ext=bfly_viol,
        cal_viol_pct_ext=cal_viol,
        blind_bfly=bool(pen_bfly <= BLIND_TOL and bfly_viol > 0.0),
        blind_cal=bool(pen_cal <= BLIND_TOL and cal_viol > 0.0),
    )


def report_blind_spots(tag, c):
    """Print a loud warning when a collocation grid is lying to the optimizer."""
    msgs = []
    if c["blind_cal"]:
        msgs.append(f"CALENDAR: pen_cal={c['pen_cal']:.2e} at the nodes but "
                    f"{c['cal_viol_pct_ext']:.1f}% of the audit grid violates "
                    f"(min d_tau w = {c['min_cal_ext']:+.4f})")
    if c["blind_bfly"]:
        msgs.append(f"BUTTERFLY: pen_bfly={c['pen_bfly']:.2e} at the nodes but "
                    f"{c['bfly_viol_pct_ext']:.1f}% of the audit grid violates "
                    f"(min g = {c['min_g_ext']:+.4f})")
    for m in msgs:
        print(f"  !! BLIND SPOT [{tag}] {m}\n"
              f"     -> constraint satisfied WHERE SAMPLED, violated BETWEEN samples.\n"
              f"     -> collocation-resolution failure; densify the relevant grid.")
    return len(msgs) == 0


def train_logged(w_model, params, kq, tq, wq, wtq, G, lam_b, lam_c, epochs,
                 lr=5e-3, n_logs=12, ext_dom=None):
    loss = make_loss(w_model, kq, tq, wq, wtq, G, lam_b, lam_c)
    logs, step = [], max(1, epochs // n_logs)
    state, done = None, 0
    while done < epochs:
        e = min(step, epochs - done)
        params, state = adam(loss, params, e, lr=lr, state=state)   # state PERSISTS
        done += e
        comp = loss_components(w_model, params, kq, tq, wq, wtq, G, ext_dom)
        logs.append({"epoch": done, **comp})
    return params, logs


def two_stage_train(wm, params, kx, tx, wx, wtx, G, lb, lc, epochs, ext_dom):
    """Stress-test budget: coarse phase at lr 5e-3, refinement at 1e-3 (Adam state is reset at
    the lr change, deliberately -- standard practice). Sharp/localized features are what tanh MLPs
    learn last (spectral bias); the refinement phase is INTENDED to let the models fit the
    distortion. Whether it does is not assumed -- it is checked by the fit-validity gate in 3b."""
    e1 = epochs // 3
    params, _ = train_logged(wm, params, kx, tx, wx, wtx, G, lb, lc,
                             e1, lr=5e-3, n_logs=2, ext_dom=ext_dom)
    params, _ = train_logged(wm, params, kx, tx, wx, wtx, G, lb, lc,
                             epochs - e1, lr=1e-3, n_logs=2, ext_dom=ext_dom)
    return params



## 3. Synthetic validation with equal-epoch ablations and a λ sweep

Ground truth: a known SSVI surface; **sparse, irregular, noisy** quotes; a prior *fitted from the
quotes* (so it is deliberately misspecified and the corrector has genuine work). Four variants,
**all trained for the same number of epochs**:

| Variant | Prior | Constraints | Tests |
|---|---|---|---|
| **full** | SSVI | on | the proposed method |
| no_constraints | SSVI | off | do penalties matter? |
| no_prior | flat | on | does the prior matter? |
| no_prior_no_constraints | flat | off | the naive MLP baseline |

**How to read a "no-arbitrage-anywhere" outcome.** With an arbitrage-free truth, a well-fitted
prior and mild noise, $g$ can stay positive throughout training for *every* variant — and then
$\nabla\,\mathrm{ReLU}(-g)^2 = 0$ *exactly*: the penalized and unpenalized runs follow the **same
trajectory**. Expect the two `no_prior` rows to come out **identical to the last digit**, and the
`no_prior` trace to be invisible on the plot because it lies exactly under
`no_prior_no_constraints`. That is the finding, not a copy-paste error: **on clean data the prior
does the protective work and the penalties are dormant.**



In [60]:
# ---------- ground truth and sparse quotes ----------
TRUE = dict(rho=-0.55, eta=0.9, gamma=0.42)
theta_true = lambda t: 0.045 * t ** 0.95


def sample_quotes(n_per=(6, 14), taus=(0.06, 0.14, 0.27, 0.5, 0.9, 1.4), noise=0.015, seed=1,
                  inflate=None, bump=None):
    """inflate=(slice_idx, factor): multiply one slice (calendar stressor).
    bump=(slice_idx, amp, center, width): local Gaussian bump (butterfly stressor)."""
    r = onp.random.default_rng(seed)
    ks, ts, ws = [], [], []
    for i, t in enumerate(taus):
        n = r.integers(*n_per)
        k = onp.sort(r.uniform(-0.45, 0.3, n))
        w = ssvi_w_np(k, theta_true(t), **TRUE) * (1 + noise * r.standard_normal(n))
        if inflate is not None and i == inflate[0]:
            w = w * inflate[1]
        if bump is not None and i == bump[0]:
            w = w * (1 + bump[1] * onp.exp(-((k - bump[2]) / bump[3]) ** 2))
        ks.append(k); ts.append(onp.full(n, t)); ws.append(onp.asarray(w))
    return onp.concatenate(ks), onp.concatenate(ts), onp.concatenate(ws)


kq, tq, wq = sample_quotes()
wtq = iv_target_weights(wq, tq)
print(f"{len(kq)} sparse quotes over {len(onp.unique(tq))} maturities")


# ---------- ROBUST prior calibration (v6) ----------
def fit_prior(kq, tq, wq, wtq):
    """Weighted SSVI prior fit, robust to sparse/one-sided slices.

    v5 estimated per-slice ATM variance with onp.interp(0.0, k, w) -- which CLAMPS outside the
    quoted k-range. On a subsampled slice whose quotes all sit on one side of the money it
    silently returned a wing value (overestimated under negative skew), corrupting theta(tau)
    and hence (alpha, beta). A WRONG prior is worse than a FLAT one for a multiplicative
    corrector (14.2 vp vs 1.7 vp at 50% subsampling): the corrector must first undo it.

    Stage 1: per-slice ATM proxy = the quote NEAREST the money, kept only if |k| <= 0.20.
    Stage 2: shape parameters (rho, eta, gamma) at fixed (alpha0, beta0).
    Stage 3: joint Nelder-Mead over ALL FIVE parameters -- the full weighted SSE pools every
             quote to re-estimate theta, instead of one fragile point per slice."""
    taus = onp.unique(tq)

    th_t, th_v = [], []
    for t in taus:
        kk_, ww_ = kq[tq == t], wq[tq == t]
        if len(kk_) == 0:
            continue
        j = int(onp.argmin(onp.abs(kk_)))
        if abs(float(kk_[j])) <= 0.20 and float(ww_[j]) > 0:
            th_t.append(float(t)); th_v.append(float(ww_[j]))
    if len(th_t) >= 2:
        A = onp.vstack([onp.ones(len(th_t)), onp.log(th_t)]).T
        coef, *_ = onp.linalg.lstsq(A, onp.log(th_v), rcond=None)
        alpha0 = float(onp.exp(coef[0]))
        beta0 = float(onp.clip(coef[1], 0.3, 1.5))
    else:  # degenerate subsample: global power-law fallback
        alpha0 = float(onp.median(onp.asarray(wq) / onp.asarray(tq) ** 0.95))
        beta0 = 0.95

    def sse(rho, eta, gamma, alpha, beta):
        rho   = float(onp.clip(rho, -0.95, 0.95))
        eta   = float(max(eta, 1e-4))
        gamma = float(onp.clip(gamma, 0.05, 0.95))
        alpha = float(max(alpha, 1e-6))
        beta  = float(onp.clip(beta, 0.3, 1.5))
        tot = 0.0
        for t in taus:
            m = tq == t
            r_ = ssvi_w_np(kq[m], alpha * t ** beta, rho, eta, gamma) - wq[m]
            tot += float(onp.sum(wtq[m] * r_ * r_))
        return tot

    best = None
    for x0 in [(-0.5, 1.0, 0.3), (-0.7, 0.6, 0.4)]:
        r = minimize(lambda p: sse(p[0], p[1], p[2], alpha0, beta0), x0, method="Nelder-Mead")
        if best is None or r.fun < best.fun:
            best = r
    rho1, eta1, gamma1 = best.x

    r = minimize(lambda p: sse(*p), [rho1, eta1, gamma1, alpha0, beta0],
                 method="Nelder-Mead", options={"maxiter": 800, "xatol": 1e-7, "fatol": 1e-14})
    rho, eta, gamma, alpha, beta = r.x
    return dict(rho=float(onp.clip(rho, -0.95, 0.95)), eta=float(max(eta, 1e-4)),
                gamma=float(onp.clip(gamma, 0.05, 0.95)),
                alpha=float(max(alpha, 1e-6)), beta=float(onp.clip(beta, 0.3, 1.5)))


def prior_rmse_iv(prior_fn, kx, tx, wx):
    """Prior-only fit quality in vol points: the corrector STARTS here. Under sparsity this
    number is the diagnostic that separates prior-calibration failures from corrector failures."""
    w_hat = onp.asarray(prior_fn(np.array(kx), np.array(tx)))
    return float(onp.sqrt(onp.mean((onp.sqrt(onp.maximum(w_hat, 1e-12) / tx)
                                    - onp.sqrt(wx / tx)) ** 2))) * 100


prior_p = fit_prior(kq, tq, wq, wtq)
print("fitted prior:", {k: round(v, 4) for k, v in prior_p.items()})
prior = make_prior(**prior_p)
prior_flat = make_prior(rho=0.0, eta=1e-6, gamma=0.3, alpha=prior_p["alpha"], beta=prior_p["beta"])
print(f"prior-only RMSE(IV, quotes) = {prior_rmse_iv(prior, kq, tq, wq):.3f} vp "
      f"(the corrector starts here)")

# ---------- collocation + scaling + domains ----------
k_lo, k_hi = float(kq.min()), float(kq.max())
t_lo, t_hi = float(tq.min()), float(tq.max())
GRIDS = make_collocation(k_lo, k_hi, t_lo, t_hi, verbose=True)

E_LO, E_HI = ext_range(k_lo, k_hi)
EXT_DOM = (E_LO, E_HI, t_lo, t_hi)
K_SC = max(abs(E_LO), abs(E_HI))
T_MID, T_SC = (t_lo + t_hi) / 2, (t_hi - t_lo) / 2
print(f"quoted k [{k_lo:+.3f},{k_hi:+.3f}] -> audited k [{E_LO:+.3f},{E_HI:+.3f}] | "
      f"tau [{t_lo:.3f},{t_hi:.3f}] | max tau gap bfly {GRIDS['gap_bfly']:.3f}y / "
      f"cal {GRIDS['gap_cal']:.3f}y")



54 sparse quotes over 6 maturities
fitted prior: {'rho': -0.5544, 'eta': 0.9307, 'gamma': 0.4123, 'alpha': 0.045, 'beta': 0.9472}
prior-only RMSE(IV, quotes) = 0.154 vp (the corrector starts here)
  bfly colloc: 624 pts (k in [-0.889,+0.571], 26 tau nodes, max gap 0.103y) | calendar colloc: 1440 pts (80 tau nodes, max gap 0.017y) | wings 104 pts
quoted k [-0.445,+0.286] -> audited k [-0.889,+0.571] | tau [0.060,1.400] | max tau gap bfly 0.103y / cal 0.017y


In [61]:
# ---------- dense ground truth AT THE QUOTED MATURITIES (fair to per-slice models) ----------
def dense_truth(nk=45):
    kg = onp.linspace(kq.min(), kq.max(), nk)
    tg = onp.unique(tq)
    KK, TT = onp.meshgrid(kg, tg)
    WW = onp.asarray(ssvi_w_np(KK, theta_true(TT), **TRUE))
    return KK, TT, WW


KK, TT, WW = dense_truth()


def rmse_iv_on_grid(w_model, params, KK, TT, WW):
    w_hat = onp.asarray(
        w_model(params, np.array(KK.ravel()), np.array(TT.ravel()))).reshape(KK.shape)
    return float(onp.sqrt(onp.mean((onp.sqrt(onp.maximum(w_hat, 1e-12) / TT)
                                    - onp.sqrt(WW / TT)) ** 2)))


# ---------- four variants, SAME epoch budget ----------
variants = {}
t0 = time.time()
for name, pr, lb, lc in [
    ("full",                    prior,      LAMBDA_BFLY, LAMBDA_CAL),
    ("no_constraints",          prior,      0.0,         0.0),
    ("no_prior",                prior_flat, LAMBDA_BFLY, LAMBDA_CAL),
    ("no_prior_no_constraints", prior_flat, 0.0,         0.0),
]:
    wm = make_model(pr, K_SC, T_MID, T_SC)
    p_fit, logs = train_logged(wm, init_mlp([2, *HIDDEN, 1], seed=SEED), kq, tq, wq, wtq,
                               GRIDS, lb, lc, EPOCHS_ABL, ext_dom=EXT_DOM)
    comp = loss_components(wm, p_fit, kq, tq, wq, wtq, GRIDS, EXT_DOM)
    variants[name] = dict(model=wm, params=p_fit, logs=logs,
                          rmse_iv=rmse_iv_on_grid(wm, p_fit, KK, TT, WW), **comp)
    print(f"[{name:>24}] RMSE(IV,truth) {variants[name]['rmse_iv']*100:6.3f} vp | "
          f"min g nodes {comp['min_g']:+.4f} / audit {comp['min_g_ext']:+.4f} | "
          f"min d_tau w audit {comp['min_cal_ext']:+.2e} | "
          f"viol bfly {comp['bfly_viol_pct_ext']:.1f}% cal {comp['cal_viol_pct_ext']:.1f}% "
          f"({time.time()-t0:.0f}s)")
    report_blind_spots(name, comp)



[                    full] RMSE(IV,truth)  0.096 vp | min g nodes +0.3101 / audit +0.3101 | min d_tau w audit +2.51e-02 | viol bfly 0.0% cal 0.0% (53s)
[          no_constraints] RMSE(IV,truth)  0.094 vp | min g nodes +0.3113 / audit +0.3113 | min d_tau w audit +2.42e-02 | viol bfly 0.0% cal 0.0% (108s)
[                no_prior] RMSE(IV,truth)  1.060 vp | min g nodes +0.2727 / audit +0.2682 | min d_tau w audit +1.58e-02 | viol bfly 0.0% cal 0.0% (152s)
[ no_prior_no_constraints] RMSE(IV,truth)  1.060 vp | min g nodes +0.2727 / audit +0.2682 | min d_tau w audit +1.58e-02 | viol bfly 0.0% cal 0.0% (195s)


In [62]:
# ---------- training dynamics: fit AND the extended-domain arbitrage margin ----------
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Weighted fit MSE (log)", "min g on the AUDIT domain (the honest margin)"))
cols = {"full": "#636efa", "no_constraints": "#ef553b",
        "no_prior": "#00cc96", "no_prior_no_constraints": "#ab63fa"}
for name, col in cols.items():
    L = variants[name]["logs"]
    fig.add_trace(go.Scatter(x=[l["epoch"] for l in L], y=[max(l["fit"], 1e-14) for l in L],
                             name=name, line=dict(color=col)), 1, 1)
    fig.add_trace(go.Scatter(x=[l["epoch"] for l in L], y=[l["min_g_ext"] for l in L],
                             name=name, line=dict(color=col), showlegend=False), 1, 2)
fig.update_yaxes(type="log", row=1, col=1)
fig.add_hline(y=0, line_dash="dot", row=1, col=2)
fig.update_xaxes(title_text="epoch")
fig.update_layout(
    width=980, height=390,
    title="Equal-epoch dynamics — persistent Adam state (no_prior lies EXACTLY under "
          "no_prior_no_constraints: zero penalty gradient)")
fig.show()



### 3b. Stress test: a violation that is guaranteed *and fittable*

The main ablation shows the penalties dormant on clean data; to isolate them, the data must
**demand** arbitrage *and the models must actually fit the distortion* — an unfitted stressor
produces only non-convergence noise. Both requirements shape the design:

- **Deflate the *last* slice ×0.45.** Since $0.45 < \theta(0.9)/\theta(1.4) = (0.9/1.4)^{0.95}
  \approx 0.657$, the deflated quotes sit *strictly below* the previous maturity's: any surface
  fitting both must have $\partial_\tau w < 0$ between them.
- **Deflation, not inflation.** The IV-target weights are $\propto 1/(4w\tau)$: inflating a slice
  *halves* its weight, deflating *raises* it. And the required corrector feature is a smooth
  decline toward the domain edge in $\tau$ — low-frequency, which a tanh MLP learns readily.
- **Dedicated budget** (`EPOCHS_STRESS`, two-stage lr), **own domain / collocation grids**
  recomputed on the stressed quotes, and a **resolution assertion**: the calendar grid's max node
  gap must be well below the width of the stressed interval (the v3 failure mode, structurally
  ruled out).
- The design is *intended* to make the stress fittable; whether it is is **not asserted but
  verified** by a two-condition gate on the unconstrained model: (i) small deflated-slice RMSE,
  **and** (ii) the fitted surface must itself reproduce at least half the data-level crossing —
  a small RMSE alone is not enough, since a surface can graze the slice from above without ever
  crossing.

A synthetic **butterfly** stressor is deliberately absent: at short maturities $g<0$ is driven by
the *slope* term $\tfrac{w'^2}{4w}$ (tiny $w$), which would require distortions far sharper than a
smooth prior × smooth corrector can produce at fittable widths. The butterfly penalty is instead
ablated on **real ultra-short quotes** (§5b), the dataset's natural stressor.



In [63]:
# ---------- Stress A: LAST slice deflated x0.45 -> guaranteed, fittable crossing ----------
kq_a, tq_a, wq_a = sample_quotes(noise=0.01, seed=3, inflate=(5, 0.45))
wtq_a = iv_target_weights(wq_a, tq_a)
t_un = onp.unique(tq_a)
m_prev, m_defl = tq_a == t_un[4], tq_a == t_un[5]
kk_ov = onp.linspace(max(kq_a[m_prev].min(), kq_a[m_defl].min()),
                     min(kq_a[m_prev].max(), kq_a[m_defl].max()), 50)
cross_data = float(onp.max(onp.interp(kk_ov, kq_a[m_prev], wq_a[m_prev])
                           - onp.interp(kk_ov, kq_a[m_defl], wq_a[m_defl])))
print(f"Stress A data-level crossing max(w_tau5 - w_tau6) = {cross_data:+.4f}  "
      f"(> 0: the DATA demand d_tau w < 0 between {t_un[4]:.2f}y and {t_un[5]:.2f}y)")


def rmse_iv_quotes(wm, p, kx, tx, wx, mask=None):
    if mask is not None:
        kx, tx, wx = kx[mask], tx[mask], wx[mask]
    w_hat = onp.asarray(wm(p, np.array(kx), np.array(tx)))
    return float(onp.sqrt(onp.mean((onp.sqrt(onp.maximum(w_hat, 1e-12) / tx)
                                    - onp.sqrt(wx / tx)) ** 2))) * 100


pr_a = make_prior(**fit_prior(kq_a, tq_a, wq_a, wtq_a))

# --- domain, collocation and input scaling RECOMPUTED on the STRESSED quotes ---
k_lo_a, k_hi_a = float(kq_a.min()), float(kq_a.max())
t_lo_a, t_hi_a = float(tq_a.min()), float(tq_a.max())
GRIDS_A = make_collocation(k_lo_a, k_hi_a, t_lo_a, t_hi_a, verbose=True)
E_LO_A, E_HI_A = ext_range(k_lo_a, k_hi_a)
EXT_DOM_A = (E_LO_A, E_HI_A, t_lo_a, t_hi_a)
K_SC_A = max(abs(E_LO_A), abs(E_HI_A))
T_MID_A, T_SC_A = (t_lo_a + t_hi_a) / 2, (t_hi_a - t_lo_a) / 2

# The crossing lives in [tau5, tau6]. If the CALENDAR grid cannot resolve that interval, the
# penalty is structurally blind there and the whole stress test is meaningless (the v3 bug).
CROSS_WIDTH = float(t_un[5] - t_un[4])
print(f"Stress A: crossing interval [{t_un[4]:.2f},{t_un[5]:.2f}] is {CROSS_WIDTH:.2f}y wide; "
      f"max calendar tau gap = {GRIDS_A['gap_cal']:.3f}y -> "
      f"{'CALENDAR GRID COVERED' if GRIDS_A['gap_cal'] < CROSS_WIDTH / 2 else 'UNDER-RESOLVED (the v3 bug)'}")
assert GRIDS_A["gap_cal"] < CROSS_WIDTH / 2, "calendar grid cannot resolve the stressed interval"


def model_crossing(wm, p_f):
    """Fitted-surface counterpart of cross_data: max over the overlap band of
    w_hat(tau5) - w_hat(tau6). > 0 means the fitted surface itself crosses."""
    w_prev = onp.asarray(wm(p_f, np.array(kk_ov), np.array(onp.full_like(kk_ov, t_un[4]))))
    w_defl = onp.asarray(wm(p_f, np.array(kk_ov), np.array(onp.full_like(kk_ov, t_un[5]))))
    return float(onp.max(w_prev - w_defl))


stress = {}
for name, lb, lc in [("full", LAMBDA_BFLY, LAMBDA_CAL), ("no_constraints", 0.0, 0.0)]:
    wm = make_model(pr_a, K_SC_A, T_MID_A, T_SC_A)
    p_f = two_stage_train(wm, init_mlp([2, *HIDDEN, 1], seed=SEED), kq_a, tq_a, wq_a, wtq_a,
                          GRIDS_A, lb, lc, EPOCHS_STRESS, EXT_DOM_A)
    c = loss_components(wm, p_f, kq_a, tq_a, wq_a, wtq_a, GRIDS_A, EXT_DOM_A)
    stress[name] = (wm, p_f, c)
    print(f"[{name:>15}] fit(all) {rmse_iv_quotes(wm, p_f, kq_a, tq_a, wq_a):.3f} | "
          f"fit(DEFLATED) {rmse_iv_quotes(wm, p_f, kq_a, tq_a, wq_a, m_defl):.3f} vp | "
          f"crossing {model_crossing(wm, p_f):+.4f} (data {cross_data:+.4f})")
    print(f"{'':>17} pen_cal nodes {c['pen_cal']:.3e} / audit {c['pen_cal_ext']:.3e} | "
          f"min d_tau w audit {c['min_cal_ext']:+.4f} | cal viol {c['cal_viol_pct_ext']:.1f}% | "
          f"bfly viol {c['bfly_viol_pct_ext']:.1f}%")
    report_blind_spots(name, c)



Stress A data-level crossing max(w_tau5 - w_tau6) = +0.0290  (> 0: the DATA demand d_tau w < 0 between 0.90y and 1.40y)
  bfly colloc: 624 pts (k in [-0.871,+0.548], 26 tau nodes, max gap 0.103y) | calendar colloc: 1440 pts (80 tau nodes, max gap 0.017y) | wings 104 pts
Stress A: crossing interval [0.90,1.40] is 0.50y wide; max calendar tau gap = 0.017y -> CALENDAR GRID COVERED
[           full] fit(all) 1.148 | fit(DEFLATED) 1.963 vp | crossing -0.0002 (data +0.0290)
                  pen_cal nodes 8.422e-11 / audit 1.176e-10 | min d_tau w audit -0.0001 | cal viol 1.6% | bfly viol 0.0%
  !! BLIND SPOT [full] CALENDAR: pen_cal=8.42e-11 at the nodes but 1.6% of the audit grid violates (min d_tau w = -0.0001)
     -> constraint satisfied WHERE SAMPLED, violated BETWEEN samples.
     -> collocation-resolution failure; densify the relevant grid.
[ no_constraints] fit(all) 0.166 | fit(DEFLATED) 0.069 vp | crossing +0.0296 (data +0.0290)
                  pen_cal nodes 1.350e-03 / audit 1.28

In [64]:
# --- FIT-VALIDITY GATE (two conditions). The arbitrage comparison is interpretable ONLY if the
# unconstrained model (a) fits the deflated slice and (b) actually reproduces the crossing. ---
GATE_TOL   = 1.0    # vol pts on the deflated slice
CROSS_FRAC = 0.5    # fitted crossing must be >= this fraction of the data-level crossing

wm_nc, p_nc, c_nc = stress["no_constraints"]
_rmse_nc  = rmse_iv_quotes(wm_nc, p_nc, kq_a, tq_a, wq_a, m_defl)
_cross_nc = model_crossing(wm_nc, p_nc)
gate_fit   = _rmse_nc < GATE_TOL
gate_cross = _cross_nc > CROSS_FRAC * cross_data
STRESS_VALID = gate_fit and gate_cross

print("FIT-VALIDITY GATE (no_constraints):")
print(f"  (a) deflated-slice RMSE = {_rmse_nc:7.3f} vp   (tol {GATE_TOL})            "
      f"-> {'PASS' if gate_fit else 'FAIL'}")
print(f"  (b) fitted crossing     = {_cross_nc:+7.4f}     (need > {CROSS_FRAC*cross_data:+.4f}) "
      f"-> {'PASS' if gate_cross else 'FAIL'}")
print(f"  => {'PASS: the arbitrage comparison below is meaningful' if STRESS_VALID else 'FAIL: increase EPOCHS_STRESS -- do NOT interpret the arbitrage comparison'}")

# --- THE ACTUAL CLAIM, tested rather than asserted ---
c_full = stress["full"][2]
_cross_full = model_crossing(*stress["full"][:2])
PENALTY_WORKS = (c_full["cal_viol_pct_ext"] < 0.5 * c_nc["cal_viol_pct_ext"]
                 and _cross_full < 0.5 * _cross_nc)
print("\nDOES THE CALENDAR PENALTY ACTUALLY REFUSE THE ARBITRAGE?")
print(f"  cal viol on audit grid: lam=0 {c_nc['cal_viol_pct_ext']:5.1f}%  ->  "
      f"lam=10 {c_full['cal_viol_pct_ext']:5.1f}%")
print(f"  fitted crossing:        lam=0 {_cross_nc:+.4f}  ->  "
      f"lam=10 {_cross_full:+.4f}  (data {cross_data:+.4f})")
print(f"  => {'YES: the penalty arbitrates the trade-off' if PENALTY_WORKS else 'NO: the constrained model still violates -- check the blind-spot warnings above'}")



FIT-VALIDITY GATE (no_constraints):
  (a) deflated-slice RMSE =   0.069 vp   (tol 1.0)            -> PASS
  (b) fitted crossing     = +0.0296     (need > +0.0145) -> PASS
  => PASS: the arbitrage comparison below is meaningful

DOES THE CALENDAR PENALTY ACTUALLY REFUSE THE ARBITRAGE?
  cal viol on audit grid: lam=0  34.7%  ->  lam=10   1.6%
  fitted crossing:        lam=0 +0.0296  ->  lam=10 -0.0002  (data +0.0290)
  => YES: the penalty arbitrates the trade-off


In [65]:
# ---------- Stress A figures: d_tau w maps + the fitted term structure at k=0 ----------
_title = ("Stress A (last slice x0.45): the data demand d_tau w < 0 — "
          + ("the penalty refuses" if PENALTY_WORKS else "the penalty FAILS to refuse (see warnings)"))
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "full: d_tau w (calendar penalty active)", "no constraints: d_tau w — red = calendar arbitrage"))
for j, name in enumerate(["full", "no_constraints"], start=1):
    wm, p_f, _ = stress[name]
    kg, tg, G, WT = surf_g_and_cal(wm, p_f, *EXT_DOM_A)
    fig.add_trace(go.Heatmap(z=WT, x=kg, y=tg, zmid=0, colorscale="RdBu",
                             showscale=(j == 2), colorbar=dict(title="d_tau w")), 1, j)
    fig.add_trace(viol_overlay(WT, kg, tg), 1, j)
fig.update_xaxes(title_text="log-moneyness k")
fig.update_yaxes(title_text="tau", col=1)
fig.update_layout(width=1000, height=430, title=_title)
fig.show()
print(f"Calendar penalty enforced on a {CAL_NT}-node uniform tau grid "
      f"(max gap {GRIDS_A['gap_cal']:.3f}y) — denser than this plot's own resolution.\n"
      f"The red overlay flags ANY negative value: at lam=10 the deepest residual is "
      f"d_tau w = {c_full['min_cal_ext']:+.1e} — numerically nonzero, economically nil.\n"
      f"Soft constraints REDUCE violations; they do not abolish them.")

# ATM term structure: quotes vs both fits
tt_line = onp.linspace(t_lo_a, t_hi_a, 200)
fig = go.Figure()
for name, col in [("full", "#636efa"), ("no_constraints", "#ef553b")]:
    wm, p_f, _ = stress[name]
    w_line = onp.asarray(wm(p_f, np.array(onp.zeros_like(tt_line)), np.array(tt_line)))
    fig.add_trace(go.Scatter(x=tt_line, y=w_line, name=name, line=dict(color=col)))
th_q = onp.array([float(onp.interp(0.0, kq_a[tq_a == t], wq_a[tq_a == t])) for t in t_un])
fig.add_trace(go.Scatter(x=t_un, y=th_q, mode="markers", name="ATM quotes (deflated last)",
                         marker=dict(color="black", size=9, symbol="x")))
fig.update_layout(width=880, height=430, xaxis_title="tau (years)",
                  yaxis_title="ATM total variance w(0, tau)",
                  title="The arbitration at k=0: no_constraints follows the deflated quotes down; "
                        "full flattens and refuses")
fig.show()



Calendar penalty enforced on a 80-node uniform tau grid (max gap 0.017y) — denser than this plot's own resolution.
The red overlay flags ANY negative value: at lam=10 the deepest residual is d_tau w = -1.3e-04 — numerically nonzero, economically nil.
Soft constraints REDUCE violations; they do not abolish them.


In [66]:
# ---------- DIAGNOSTIC (keep this: it is what found the v3 bug) ----------
wm_nc, p_nc, _ = stress["no_constraints"]
kcal_a, tcal_a = GRIDS_A["kcal"], GRIDS_A["tcal"]

pen_cal_only = lambda p: np.mean(relu(-egrad(lambda t: wm_nc(p, kcal_a, t))(tcal_a)) ** 2)
print(f"pen_cal at the VIOLATING optimum = {float(pen_cal_only(p_nc)):.6e}")
gr = grad(pen_cal_only)(p_nc)
gnorm = sum(float(onp.abs(gW).sum() + onp.abs(gb).sum()) for gW, gb in gr)
print(f"||grad pen_cal|| at that point   = {gnorm:.6e}   <-- if ~0, the AD path is broken")

L10 = make_loss(wm_nc, kq_a, tq_a, wq_a, wtq_a, GRIDS_A, LAMBDA_BFLY, LAMBDA_CAL)
for nm in ("full", "no_constraints"):
    _, p, c = stress[nm]
    print(f"[{nm:>15}] total loss(lam=10) = {float(L10(p)):.6e} | fit = {c['fit']:.3e} | "
          f"pen_cal nodes = {c['pen_cal']:.3e} | pen_cal audit = {c['pen_cal_ext']:.3e}")
print("\nReading: if `full` has the LOWER total loss AND pen_cal(nodes) ~ 0 AND pen_cal(audit) >> 0,\n"
      "the optimizer is not at fault and the AD is not at fault -- the GRID is. That was v3.\n"
      "With the dedicated calendar grid, nodes and audit must now agree to within grid noise.")



pen_cal at the VIOLATING optimum = 1.349724e-03
||grad pen_cal|| at that point   = 3.783746e-01   <-- if ~0, the AD path is broken
[           full] total loss(lam=10) = 5.601206e-07 | fit = 5.479e-07 | pen_cal nodes = 8.422e-11 | pen_cal audit = 1.176e-10
[ no_constraints] total loss(lam=10) = 1.349726e-02 | fit = 1.162e-08 | pen_cal nodes = 1.350e-03 | pen_cal audit = 1.288e-03

Reading: if `full` has the LOWER total loss AND pen_cal(nodes) ~ 0 AND pen_cal(audit) >> 0,
the optimizer is not at fault and the AD is not at fault -- the GRID is. That was v3.
With the dedicated calendar grid, nodes and audit must now agree to within grid noise.


### 3c. λ sweep on the stressed data (Ackerer Fig. 2 / Table 1 protocol)

$\lambda=0$ and $\lambda=10$ reuse the two stress runs above; only $\lambda=1$ is trained here.
Expected pattern: $\lambda=0$ fits the deflated slice and violates; increasing $\lambda$ buys the
violation back at a measurable price in the **deflated-slice fit**.

The sweep is only interpretable if (i) the fit-validity gate passed **and** (ii) no blind-spot
warning fired — a monotone-looking sweep on an under-resolved grid measures nothing.



In [67]:
wm1 = make_model(pr_a, K_SC_A, T_MID_A, T_SC_A)
p1 = two_stage_train(wm1, init_mlp([2, *HIDDEN, 1], seed=SEED), kq_a, tq_a, wq_a, wtq_a,
                     GRIDS_A, 1.0, 1.0, EPOCHS_STRESS, EXT_DOM_A)
c1 = loss_components(wm1, p1, kq_a, tq_a, wq_a, wtq_a, GRIDS_A, EXT_DOM_A)

sweep = [(0.0, stress["no_constraints"]), (1.0, (wm1, p1, c1)), (10.0, stress["full"])]
if not STRESS_VALID:
    print("[warning] fit-validity gate FAILED -- the sweep below is not interpretable.\n")
for lam, (wm, p_f, c) in sweep:
    print(f"lambda={lam:>4.0f} | fit(DEFLATED) {rmse_iv_quotes(wm, p_f, kq_a, tq_a, wq_a, m_defl):6.3f} vp | "
          f"crossing {model_crossing(wm, p_f):+.4f} | "
          f"pen_cal nodes {c['pen_cal']:.2e} audit {c['pen_cal_ext']:.2e} | "
          f"min d_tau w {c['min_cal_ext']:+.4f} | cal viol {c['cal_viol_pct_ext']:5.1f}% | "
          f"bfly viol {c['bfly_viol_pct_ext']:4.1f}%")
    report_blind_spots(f"lam={lam:.0f}", c)



lambda=   0 | fit(DEFLATED)  0.069 vp | crossing +0.0296 | pen_cal nodes 1.35e-03 audit 1.29e-03 | min d_tau w -0.1478 | cal viol  34.7% | bfly viol  0.0%
lambda=   1 | fit(DEFLATED)  1.991 vp | crossing +0.0000 | pen_cal nodes 1.30e-09 audit 1.22e-09 | min d_tau w -0.0004 | cal viol   4.4% | bfly viol  0.0%
lambda=  10 | fit(DEFLATED)  1.963 vp | crossing -0.0002 | pen_cal nodes 8.42e-11 audit 1.18e-10 | min d_tau w -0.0001 | cal viol   1.6% | bfly viol  0.0%
  !! BLIND SPOT [lam=10] CALENDAR: pen_cal=8.42e-11 at the nodes but 1.6% of the audit grid violates (min d_tau w = -0.0001)
     -> constraint satisfied WHERE SAMPLED, violated BETWEEN samples.
     -> collocation-resolution failure; densify the relevant grid.


In [68]:
# ---------- final full model (longer run) + reconstructed smiles vs truth ----------
wm_full = make_model(prior, K_SC, T_MID, T_SC)
p_full, _ = train_logged(wm_full, init_mlp([2, *HIDDEN, 1], seed=SEED), kq, tq, wq, wtq,
                         GRIDS, LAMBDA_BFLY, LAMBDA_CAL, EPOCHS_MAIN,
                         n_logs=6, ext_dom=EXT_DOM)
print(f"final full model: RMSE(IV, truth) = "
      f"{rmse_iv_on_grid(wm_full, p_full, KK, TT, WW)*100:.3f} vol pts")

fig = go.Figure()
palette = ["#636efa", "#ef553b", "#00cc96", "#ab63fa", "#ffa15a", "#19d3f3"]
for t, c in zip(onp.unique(tq), palette):
    kk_ = onp.linspace(kq.min(), kq.max(), 120)
    iv_hat = onp.sqrt(onp.maximum(
        onp.asarray(wm_full(p_full, np.array(kk_), np.array(onp.full_like(kk_, t)))), 1e-12) / t)
    iv_true = onp.sqrt(onp.asarray(ssvi_w_np(kk_, theta_true(t), **TRUE)) / t)
    msk = tq == t
    fig.add_trace(go.Scatter(x=kq[msk], y=onp.sqrt(wq[msk] / t), mode="markers",
                             marker=dict(color=c, size=6), name=f"{t*365:.0f}d quotes"))
    fig.add_trace(go.Scatter(x=kk_, y=iv_hat, line=dict(color=c), showlegend=False))
    fig.add_trace(go.Scatter(x=kk_, y=iv_true, line=dict(color=c, dash="dot"), showlegend=False))
fig.update_layout(width=900, height=450, xaxis_title="log-moneyness k", yaxis_title="implied vol",
                  title="Deep smoother (solid) vs ground truth (dotted) on sparse noisy quotes")
fig.show()



final full model: RMSE(IV, truth) = 0.073 vol pts


## 4. Robustness to quote sparsity — a fair, multi-seed comparison

Quotes are subsampled (100% → 50% → 25%) and the deep smoother is compared against **per-slice
SVI** on the dense truth restricted to the quoted maturities.

**Three defects fixed across v5/v6:**

1. **Prior leakage** (fixed in v5): the prior/collocation/scaling were calibrated on the full
   quote set inside the subsampling loop — the deep model kept seeing quotes SVI was denied,
   which is why its RMSE *fell* as data was removed. Everything is now refit per subsample.
2. **Prior-calibration instability** (fixed in v6): with the leak removed, one 50% subsample
   produced 14.2 vp — the naive ATM-interp in `fit_prior` clamped on one-sided slices and
   corrupted θ(τ). `fit_prior` is now robust (nearest-ATM proxy + joint 5-parameter refinement),
   and the **prior-only RMSE is printed per run**: if a run still blows up, that column says
   whether the prior or the corrector failed.
3. **Single-seed lottery** (fixed in v6): one subsample draw is one lottery ticket (the old 25%
   run beat the old 50% run purely by draw). Each fraction now runs `SPARSITY_SEEDS` draws;
   the figure shows individual draws and the median.

**Non-comparable denominators** (fixed in v5, kept): `svi_surface_rmse` skips slices with
`< MIN_PTS_SLICE` points, so its RMSE is *conditional on calibrability*. The deep model is also
reported on **SVI's calibrable slices only**, and coverage is reported separately.

At 25% the ~14 surviving quotes average ~2.3 points/slice: no slice reaches 6, SVI returns `n/a`.
That is a **coverage** failure — the point of the section — not an accuracy one.

**The honest headline:** the deep smoother's sparsity robustness is *inherited from the prior's
calibration robustness*. The prior×corrector architecture moves the failure point; it does not
remove it.



In [69]:
# --- NB02 quasi-explicit SVI (compact copy, the parametric contender) ---
def _svi_inner(m, s, k, w, wt):
    y = (k - m) / s; z = onp.sqrt(y * y + 1.0)
    A = onp.column_stack([onp.ones_like(y), y, z])
    wmax = float(max(w.max(), 1e-6))
    cons = [{"type": "ineq", "fun": lambda x: x[2]},
            {"type": "ineq", "fun": lambda x: 4 * s - x[2]},
            {"type": "ineq", "fun": lambda x: x[2] - x[1]},
            {"type": "ineq", "fun": lambda x: x[2] + x[1]},
            {"type": "ineq", "fun": lambda x: (4 * s - x[2]) - x[1]},
            {"type": "ineq", "fun": lambda x: x[1] - (x[2] - 4 * s)},
            {"type": "ineq", "fun": lambda x: x[0]},
            {"type": "ineq", "fun": lambda x: wmax - x[0]}]
    r = minimize(lambda x: float(onp.sum(wt * (A @ x - w) ** 2)),
                 onp.array([onp.median(w), 0.0, min(2 * s, wmax)]),
                 jac=lambda x: 2.0 * A.T @ (wt * (A @ x - w)), method="SLSQP", constraints=cons,
                 options={"maxiter": 200, "ftol": 1e-14})
    return r.x, r.fun


def fit_svi_slice_np(k, w, wt):
    best = None
    for m0, s0 in ((0.0, 0.1), (0.0, 0.2)):
        r = minimize(lambda ms: _svi_inner(ms[0], onp.exp(ms[1]), k, w, wt)[1], [m0, onp.log(s0)],
                     method="Nelder-Mead", options={"maxiter": 300})
        x, f_ = _svi_inner(r.x[0], float(onp.exp(r.x[1])), k, w, wt)
        if best is None or f_ < best[0]:
            a, d, c = x; b = c / float(onp.exp(r.x[1]))
            rho = float(onp.clip(d / c, -0.999, 0.999)) if c > 1e-12 else 0.0
            best = (f_, dict(a=float(a), b=float(b), rho=rho,
                             m=float(r.x[0]), s=float(onp.exp(r.x[1]))))
    return best[1]


def svi_surface_rmse(kq, tq, wq, wtq):
    """Per-slice SVI at the QUOTED maturities of the truth grid; skips slices < MIN_PTS.
    Returns (rmse, covered_row_indices) so the deep model can be scored on the SAME slices:
    SVI's RMSE is CONDITIONAL on calibrability and is not comparable to a 6-slice average."""
    errs, covered_idx = [], []
    for i, t in enumerate(TT[:, 0]):
        msk = onp.isclose(tq, t)
        if msk.sum() < MIN_PTS_SLICE:
            continue
        p = fit_svi_slice_np(kq[msk], wq[msk], wtq[msk])
        w_hat = onp.asarray(svi_raw(np.array(KK[i]), **p))
        errs.append(onp.sqrt(onp.maximum(w_hat, 1e-12) / t) - onp.sqrt(WW[i] / t))
        covered_idx.append(i)
    if not errs:
        return onp.nan, covered_idx
    return float(onp.sqrt(onp.mean(onp.concatenate(errs) ** 2))), covered_idx


rows = []
for frac in (1.0, 0.5, 0.25):
    seeds = (SPARSITY_SEEDS[0],) if frac == 1.0 else SPARSITY_SEEDS   # frac=1 keeps everything
    for sd in seeds:
        r = onp.random.default_rng(sd)
        keep = r.random(len(kq)) < frac
        kq_f, tq_f, wq_f = kq[keep], tq[keep], wq[keep]
        wtq_f = iv_target_weights(wq_f, tq_f)

        # CRITICAL: prior, collocation and input scaling refit on the SUBSAMPLE (no leakage).
        pr_p_f = fit_prior(kq_f, tq_f, wq_f, wtq_f)
        pr_f = make_prior(**pr_p_f)
        prior_only = prior_rmse_iv(pr_f, kq_f, tq_f, wq_f)

        klo_f, khi_f = float(kq_f.min()), float(kq_f.max())
        tlo_f, thi_f = float(tq_f.min()), float(tq_f.max())
        G_f = make_collocation(klo_f, khi_f, tlo_f, thi_f)
        elo_f, ehi_f = ext_range(klo_f, khi_f)
        ext_f = (elo_f, ehi_f, tlo_f, thi_f)

        wm = make_model(pr_f, max(abs(elo_f), abs(ehi_f)),
                        (tlo_f + thi_f) / 2, (thi_f - tlo_f) / 2)
        p_fit, _ = train_logged(wm, init_mlp([2, *HIDDEN, 1], seed=SEED), kq_f, tq_f, wq_f, wtq_f,
                                G_f, LAMBDA_BFLY, LAMBDA_CAL, EPOCHS_ABL,
                                n_logs=3, ext_dom=ext_f)

        deep_all = rmse_iv_on_grid(wm, p_fit, KK, TT, WW)
        svi_rmse, cov_idx = svi_surface_rmse(kq_f, tq_f, wq_f, wtq_f)
        ncov = len(cov_idx)
        deep_cov = (rmse_iv_on_grid(wm, p_fit, KK[cov_idx], TT[cov_idx], WW[cov_idx])
                    if ncov else None)

        rows.append(dict(frac=frac, seed=sd, n=int(keep.sum()),
                         prior_rmse=prior_only,
                         deep=deep_all * 100,
                         deep_cov=deep_cov * 100 if ncov else None,
                         svi=svi_rmse * 100 if ncov else None,
                         svi_slices=ncov))
        svi_txt = f"{svi_rmse*100:6.3f}" if ncov else "   n/a"
        cov_txt = f"{deep_cov*100:6.3f}" if ncov else "   n/a"
        print(f"kept {frac*100:>4.0f}% seed {sd:>2} (n={keep.sum():>3}) | "
              f"prior-only {prior_only:6.3f} | deep[all] {deep_all*100:6.3f} | "
              f"deep[SVI's {ncov}] {cov_txt} | SVI {svi_txt} vp | "
              f"{ncov}/{len(TT[:,0])} calibrable")

print("\nReading guide: if a deep run is bad AND its prior-only RMSE is bad, the PRIOR calibration\n"
      "failed on that draw (not the corrector). Deep degradation should be graceful in the MEDIAN;\n"
      "single draws fluctuate because subsampling changes both information AND noise.")

# ---------- figure: individual draws + medians ----------
def _med(vals):
    v = [x for x in vals if x is not None]
    return float(onp.median(v)) if v else None


fracs = sorted({r_["frac"] for r_ in rows}, reverse=True)
fr_pct = [f * 100 for f in fracs]
med_deep = [_med([r_["deep"] for r_ in rows if r_["frac"] == f]) for f in fracs]
med_cov  = [_med([r_["deep_cov"] for r_ in rows if r_["frac"] == f]) for f in fracs]
med_svi  = [_med([r_["svi"] for r_ in rows if r_["frac"] == f]) for f in fracs]
med_pri  = [_med([r_["prior_rmse"] for r_ in rows if r_["frac"] == f]) for f in fracs]

fig = go.Figure()
fig.add_trace(go.Scatter(x=[r_["frac"] * 100 for r_ in rows], y=[r_["deep"] for r_ in rows],
                         mode="markers", name="deep (individual draws)",
                         marker=dict(color="#636efa", size=7, opacity=0.45)))
fig.add_trace(go.Scatter(x=[r_["frac"] * 100 for r_ in rows], y=[r_["svi"] for r_ in rows],
                         mode="markers", name="SVI (individual draws)",
                         marker=dict(color="#ef553b", size=7, opacity=0.45)))
fig.add_trace(go.Scatter(x=fr_pct, y=med_deep, mode="lines+markers",
                         name="deep — median (all 6 slices)", line=dict(color="#636efa", width=3)))
fig.add_trace(go.Scatter(x=fr_pct, y=med_cov, mode="lines+markers",
                         name="deep — median (SVI's slices)",
                         line=dict(color="#636efa", dash="dot")))
fig.add_trace(go.Scatter(x=fr_pct, y=med_svi, mode="lines+markers",
                         name="per-slice SVI — median", line=dict(color="#ef553b", width=3)))
fig.add_trace(go.Scatter(x=fr_pct, y=med_pri, mode="lines+markers",
                         name="prior-only — median (deep starts here)",
                         line=dict(color="#00cc96", dash="dash")))
for f in fracs:
    ncovs = [r_["svi_slices"] for r_ in rows if r_["frac"] == f]
    fig.add_annotation(x=f * 100, y=0, yref="paper", yanchor="bottom", showarrow=False,
                       text=f"SVI: {int(onp.median(ncovs))}/6 slices")
fig.update_layout(width=880, height=460, xaxis_title="% of quotes kept",
                  yaxis_title="RMSE at quoted maturities (vol points)",
                  title="Sparsity: SVI loses COVERAGE; the surface model's accuracy is bounded "
                        "by prior calibration")
fig.update_xaxes(autorange="reversed")
fig.show()



kept  100% seed  7 (n= 54) | prior-only  0.154 | deep[all]  0.096 | deep[SVI's 6]  0.096 | SVI  0.831 vp | 6/6 calibrable
kept   50% seed  7 (n= 26) | prior-only  0.168 | deep[all]  0.221 | deep[SVI's 1]  0.137 | SVI  1.615 vp | 1/6 calibrable
kept   50% seed 17 (n= 28) | prior-only  0.148 | deep[all]  0.109 | deep[SVI's 1]  0.091 | SVI  0.282 vp | 1/6 calibrable
kept   50% seed 27 (n= 29) | prior-only  0.174 | deep[all]  0.236 | deep[SVI's 2]  0.147 | SVI  0.251 vp | 2/6 calibrable
kept   25% seed  7 (n= 14) | prior-only  0.087 | deep[all]  0.341 | deep[SVI's 0]    n/a | SVI    n/a vp | 0/6 calibrable
kept   25% seed 17 (n= 13) | prior-only  1.207 | deep[all]  1.681 | deep[SVI's 0]    n/a | SVI    n/a vp | 0/6 calibrable
kept   25% seed 27 (n= 14) | prior-only  0.142 | deep[all]  0.225 | deep[SVI's 0]    n/a | SVI    n/a vp | 0/6 calibrable

Reading guide: if a deep run is bad AND its prior-only RMSE is bad, the PRIOR calibration
failed on that draw (not the corrector). Deep degradati

## 5. Real SPX data: day-by-day driver (NB02 protocol)

Per day: OTM quotes grouped by `exdate`; per-slice 20% hold-out seeded with `crc32(date)`;
symmetric IV-target-weighted fit; penalties on the day's constraint-specific grids; arbitrage
audited on the quoted **and** capped-extended domains; maturity-bucket RMSE.

On real SPX the deep-OTM put quotes reach $k \approx -2.2$ (a strike at ~11% of spot). `ext_range`
caps the audit at $|k| \le 1.5$ **but never inside the quoted range**, so the audit runs over
$[-2.2, \ldots]$ rather than $[-4.4, \ldots]$ — where a "violation" concerns a strike at 1% of
spot and carries no economic content.

> The bar: deep must clearly beat **SSVI** (its structural peer: one model per day) and approach
> **SVI** (5 parameters per slice, no cross-maturity coherence).



In [70]:
def run_real_day(day_df, date, seed=None, lam_b=LAMBDA_BFLY, lam_c=LAMBDA_CAL,
                 epochs=EPOCHS_REAL):
    seed = stable_seed(date) if seed is None else seed
    r = onp.random.default_rng(seed)
    ks, ts, ws, hs = [], [], [], []
    exds = sorted(day_df["exdate"].unique().to_list())
    for exd in exds:
        s = day_df.filter(plr.col("exdate") == exd).sort("k")
        k = s["k"].to_numpy(); iv = s["iv_om"].to_numpy(); tau = float(s["tau"][0])
        ok = onp.isfinite(k) & onp.isfinite(iv) & (iv > 0)
        k, iv = k[ok], iv[ok]
        if len(k) < MIN_PTS_SLICE + 2:
            continue
        w = iv ** 2 * tau
        hold = r.random(len(k)) < HOLDOUT_FRAC
        if (~hold).sum() < MIN_PTS_SLICE:
            hold[:] = False
        ks.append(k); ts.append(onp.full(len(k), tau)); ws.append(w); hs.append(hold)
    if not ks:
        return None

    k_all = onp.concatenate(ks); t_all = onp.concatenate(ts)
    w_all = onp.concatenate(ws); hold = onp.concatenate(hs)
    kq_, tq_, wq_ = k_all[~hold], t_all[~hold], w_all[~hold]
    kh, th, wh = k_all[hold], t_all[hold], w_all[hold]
    wtq_ = iv_target_weights(wq_, tq_)

    pr = make_prior(**fit_prior(kq_, tq_, wq_, wtq_))
    klo, khi = float(k_all.min()), float(k_all.max())
    tlo, thi = float(t_all.min()), float(t_all.max())
    G = make_collocation(klo, khi, tlo, thi)
    elo, ehi = ext_range(klo, khi)
    ext_d = (elo, ehi, tlo, thi)
    wm = make_model(pr, max(abs(elo), abs(ehi)), (tlo + thi) / 2, (thi - tlo) / 2)

    t_start = time.time()
    p_fit, _ = train_logged(wm, init_mlp([2, *HIDDEN, 1], seed=seed % 2 ** 31),
                            kq_, tq_, wq_, wtq_, G, lam_b, lam_c,
                            epochs, n_logs=4, ext_dom=ext_d)
    runtime = time.time() - t_start
    comp = loss_components(wm, p_fit, kq_, tq_, wq_, wtq_, G, ext_d)

    def iv_rmse(kx, tx, wx):
        if len(kx) == 0:
            return None
        w_hat = onp.asarray(wm(p_fit, np.array(kx), np.array(tx)))
        return float(onp.sqrt(onp.mean((onp.sqrt(onp.maximum(w_hat, 1e-12) / tx)
                                        - onp.sqrt(wx / tx)) ** 2)))

    edges = [(0, 14, "b_0714"), (14, 60, "b_1560"), (60, 180, "b_61180"), (180, 10000, "b_180p")]
    buckets = {}
    for lo, hi, nm in edges:
        m = (tq_ * 365 > lo) & (tq_ * 365 <= hi)
        buckets[nm] = iv_rmse(kq_[m], tq_[m], wq_[m]) if m.sum() >= 4 else None

    return dict(n=len(k_all), rmse_in=iv_rmse(kq_, tq_, wq_), rmse_hold=iv_rmse(kh, th, wh),
                min_g=comp["min_g"], min_g_ext=comp["min_g_ext"], min_cal_ext=comp["min_cal_ext"],
                pen_bfly=comp["pen_bfly"], pen_cal=comp["pen_cal"],
                pen_bfly_ext=comp["pen_bfly_ext"], pen_cal_ext=comp["pen_cal_ext"],
                bfly_viol_pct_ext=comp["bfly_viol_pct_ext"],
                cal_viol_pct_ext=comp["cal_viol_pct_ext"],
                blind_bfly=comp["blind_bfly"], blind_cal=comp["blind_cal"],
                tau_gap_bfly=G["gap_bfly"], tau_gap_cal=G["gap_cal"],
                runtime_s=runtime, buckets=buckets, model=(wm, p_fit),
                grid=(klo, khi, tlo, thi), ext_dom=ext_d, comp=comp)



In [71]:
if not REAL_PARQUET.exists():
    print(f"[info] {REAL_PARQUET} not found — real-data sections skipped. Run NB01 first.")
    real_rows, last_day, df = None, None, None
else:
    df = (plr.scan_parquet(REAL_PARQUET).filter(plr.col("is_otm"))
            .select(["date", "exdate", "tau", "k", "iv_om"])
            .drop_nulls().collect(engine="streaming"))
    dates = df["date"].unique().sort().to_list()
    if LIMIT_DATES:
        dates = dates[:LIMIT_DATES]
    real_rows, last_day = [], None
    for d in dates:
        res = run_real_day(df.filter(plr.col("date") == d), d)
        if res is None:
            continue
        real_rows.append({
            "date": d, "n_quotes": res["n"],
            "deep_rmse_in": res["rmse_in"], "deep_rmse_holdout": res["rmse_hold"],
            "deep_min_g": res["min_g"], "deep_min_g_ext": res["min_g_ext"],
            "deep_min_cal_ext": res["min_cal_ext"],
            "deep_bfly_viol_pct_ext": res["bfly_viol_pct_ext"],
            "deep_cal_viol_pct_ext": res["cal_viol_pct_ext"],
            "deep_pen_bfly_nodes": res["pen_bfly"], "deep_pen_bfly_audit": res["pen_bfly_ext"],
            "deep_pen_cal_nodes": res["pen_cal"],  "deep_pen_cal_audit": res["pen_cal_ext"],
            "deep_blind_bfly": res["blind_bfly"], "deep_blind_cal": res["blind_cal"],
            "tau_gap_bfly": res["tau_gap_bfly"], "tau_gap_cal": res["tau_gap_cal"],
            "runtime_s": res["runtime_s"], **res["buckets"]})
        last_day = (d, res)
        hold_txt = f"{res['rmse_hold']*100:.3f}" if res["rmse_hold"] is not None else "n/a"
        print(f"{d}  n={res['n']:>4}  in {res['rmse_in']*100:.3f} | holdout {hold_txt} vp | "
              f"min g {res['min_g_ext']:+.4f} (bfly viol {res['bfly_viol_pct_ext']:.1f}%) | "
              f"min d_tau w {res['min_cal_ext']:+.4f} (cal viol {res['cal_viol_pct_ext']:.1f}%) | "
              f"{res['runtime_s']:.0f}s")
        report_blind_spots(str(d), res["comp"])



2018-01-02  n=3199  in 1.327 | holdout 1.306 vp | min g +0.1826 (bfly viol 0.0%) | min d_tau w +0.0035 (cal viol 0.0%) | 87s
2018-01-03  n=3356  in 1.562 | holdout 1.525 vp | min g +0.1339 (bfly viol 0.0%) | min d_tau w +0.0039 (cal viol 0.0%) | 87s


In [72]:
# --- hold-out vs hold-out comparison against the NB02 v3 benchmark, and save ---
if real_rows:
    deep_df = plr.DataFrame(real_rows, infer_schema_length=None)
    bench_slices = OUT_DIR / "benchmark_svi_slices_full.parquet"
    bench_days   = OUT_DIR / "benchmark_ssvi_days_full.parquet"
    if bench_slices.exists():
        svi = (plr.read_parquet(bench_slices).group_by("date")
                  .agg(plr.col("svi_rmse_iv_holdout").median().alias("svi_holdout_median"),
                       plr.col("svi_rmse_iv").median().alias("svi_in_median")))
        deep_df = deep_df.join(svi, on="date", how="left")
    else:
        print(f"[info] {bench_slices} not found — run NB02 v3 for the SVI comparison.")
    if bench_days.exists():
        ssvi = (plr.read_parquet(bench_days)
                   .select(["date", "ssvi_rmse_iv", "ssvi_rmse_iv_holdout"]))
        deep_df = deep_df.join(ssvi, on="date", how="left")
    else:
        print(f"[info] {bench_days} not found — run NB02 v3 for the SSVI comparison.")

    show_cols = [c for c in ["date", "n_quotes", "deep_rmse_in", "deep_rmse_holdout",
                             "svi_holdout_median", "ssvi_rmse_iv_holdout",
                             "deep_min_g_ext", "deep_bfly_viol_pct_ext",
                             "deep_min_cal_ext", "deep_cal_viol_pct_ext",
                             "deep_blind_bfly", "deep_blind_cal",
                             "tau_gap_bfly", "tau_gap_cal"] if c in deep_df.columns]
    print(deep_df.select(show_cols))
    deep_df.write_parquet(OUT_DIR / "deep_smoother_days.parquet")
    print("written:", OUT_DIR / "deep_smoother_days.parquet")



shape: (2, 14)
┌───────────┬──────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ date      ┆ n_quotes ┆ deep_rmse ┆ deep_rmse ┆ … ┆ deep_blin ┆ deep_blin ┆ tau_gap_b ┆ tau_gap_c │
│ ---       ┆ ---      ┆ _in       ┆ _holdout  ┆   ┆ d_bfly    ┆ d_cal     ┆ fly       ┆ al        │
│ date      ┆ i64      ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│           ┆          ┆ f64       ┆ f64       ┆   ┆ bool      ┆ bool      ┆ f64       ┆ f64       │
╞═══════════╪══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2018-01-0 ┆ 3199     ┆ 0.013275  ┆ 0.013061  ┆ … ┆ false     ┆ false     ┆ 0.14942   ┆ 0.024588  │
│ 2         ┆          ┆           ┆           ┆   ┆           ┆           ┆           ┆           │
│ 2018-01-0 ┆ 3356     ┆ 0.015618  ┆ 0.015248  ┆ … ┆ false     ┆ false     ┆ 0.14942   ┆ 0.024588  │
│ 3         ┆          ┆           ┆           ┆   ┆           ┆           ┆

In [73]:
# --- report figures for the last processed day: smiles + 3-D surface + audit maps ---
if real_rows and last_day is not None:
    d, res = last_day
    wm, pf = res["model"]; klo, khi, tlo, thi = res["grid"]
    day = df.filter(plr.col("date") == d)

    fig = go.Figure()
    exd = (day.group_by("exdate")
              .agg(plr.len().alias("n"), plr.col("tau").first().alias("tau"))
              .filter(plr.col("n") >= MIN_PTS_SLICE).sort("tau"))
    idx = onp.linspace(0, exd.height - 1, min(5, exd.height)).round().astype(int)
    palette = ["#636efa", "#ef553b", "#00cc96", "#ab63fa", "#ffa15a"]
    for ex, c in zip(exd["exdate"].gather(idx.tolist()), palette):
        s = day.filter(plr.col("exdate") == ex).sort("k")
        t = float(s["tau"][0])
        kk_ = onp.linspace(float(s["k"].min()), float(s["k"].max()), 120)
        ivh = onp.sqrt(onp.maximum(
            onp.asarray(wm(pf, np.array(kk_), np.array(onp.full_like(kk_, t)))), 1e-12) / t)
        fig.add_trace(go.Scatter(x=s["k"], y=s["iv_om"], mode="markers",
                                 marker=dict(color=c, size=5), name=f"{t*365:.0f}d"))
        fig.add_trace(go.Scatter(x=kk_, y=ivh, line=dict(color=c), showlegend=False))
    fig.update_layout(width=900, height=440, xaxis_title="log-moneyness k", yaxis_title="IV",
                      title=f"Deep smoother on real SPX quotes — {d}")
    fig.show()

    kg = onp.linspace(klo, khi, 50); tg = onp.linspace(tlo, thi, 25)
    Kg, Tg = onp.meshgrid(kg, tg)
    Wg = onp.asarray(wm(pf, np.array(Kg.ravel()), np.array(Tg.ravel()))).reshape(Kg.shape)
    Z = onp.sqrt(onp.maximum(Wg, 1e-12) / Tg)
    fig = go.Figure(go.Surface(x=kg, y=tg, z=Z, colorscale="Viridis", colorbar=dict(title="IV")))
    fig.update_layout(width=800, height=520,
                      scene=dict(xaxis_title="k", yaxis_title="tau", zaxis_title="IV"),
                      title=f"Deep-smoothed implied-volatility surface — {d}")
    fig.show()

    kg2, tg2, Gd, WTd = surf_g_and_cal(wm, pf, *res["ext_dom"], nk=50, nt=25)
    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        "g(k,tau) — audit domain (red < 0)", "d_tau w — audit domain (red < 0)"))
    fig.add_trace(go.Heatmap(z=Gd, x=kg2, y=tg2, zmid=0, colorscale="RdBu",
                             colorbar=dict(title="val")), 1, 1)
    fig.add_trace(viol_overlay(Gd, kg2, tg2), 1, 1)
    fig.add_trace(go.Heatmap(z=WTd, x=kg2, y=tg2, zmid=0, colorscale="RdBu",
                             showscale=False), 1, 2)
    fig.add_trace(viol_overlay(WTd, kg2, tg2), 1, 2)
    fig.update_xaxes(title_text="k"); fig.update_yaxes(title_text="tau", col=1)
    fig.update_layout(width=1000, height=420,
                      title=f"Arbitrage audit beyond the quotes (capped at |k|<={K_EXT_CAP}) — {d}")
    fig.show()



### 5b. Penalty ablation on real quotes — the natural butterfly stressor

NB02 measured that 55% of real 7–14d SVI slices are butterfly-violating: ultra-short quotes with
steep wings are where $g<0$ lives in this dataset. Refitting the last day with
$\lambda_{\text{bfly}}=\lambda_{\text{cal}}=0$ (same seed, same hold-out, same epochs) isolates
what the penalties contribute.

**Read the in-sample fit with care.** $\lambda=0$ minimises a strict subset of the loss terms, so
*at the optimum* it cannot fit worse in-sample than $\lambda=10$. If it does, that is an
**optimisation artefact**: at equal epoch budget the penalties act as a preconditioner/regulariser
and the unconstrained descent converges less well. The accuracy gap is **not** claimed as a
benefit of the constraints; only the arbitrage gap is. Set `RUN_LAM0_CHECK = True` to verify with
3× epochs and 3 seeds before writing anything about the fit numbers.



In [74]:
if real_rows and last_day is not None:
    d, res = last_day
    res0 = run_real_day(df.filter(plr.col("date") == d), d, lam_b=0.0, lam_c=0.0)
    for tag, rr in [("lambda=10 (full)", res), ("lambda=0  (none)", res0)]:
        hold_txt = f"{rr['rmse_hold']*100:.3f}" if rr["rmse_hold"] is not None else "n/a"
        print(f"[{tag}] in {rr['rmse_in']*100:.3f} | holdout {hold_txt} vp")
        print(f"{'':>18} min g {rr['min_g_ext']:+.4f} | bfly viol {rr['bfly_viol_pct_ext']:5.1f}% | "
              f"pen_bfly nodes {rr['pen_bfly']:.2e} audit {rr['pen_bfly_ext']:.2e}")
        print(f"{'':>18} min d_tau w {rr['min_cal_ext']:+.4f} | cal viol {rr['cal_viol_pct_ext']:5.1f}% | "
              f"pen_cal  nodes {rr['pen_cal']:.2e} audit {rr['pen_cal_ext']:.2e}")
        report_blind_spots(tag, rr["comp"])

    if res0["rmse_in"] < res["rmse_in"]:
        print("\n[note] lambda=0 fits BETTER in-sample, as theory requires at the optimum.")
    else:
        print(f"\n[WARNING] lambda=0 fits WORSE in-sample ({res0['rmse_in']*100:.3f} vs "
              f"{res['rmse_in']*100:.3f} vp). Impossible at the optimum: this is an OPTIMISATION\n"
              f"          artefact (equal budget; penalties precondition the descent). Do NOT\n"
              f"          report it as 'the penalties improve accuracy'. Set RUN_LAM0_CHECK=True\n"
              f"          to verify with 3x epochs and 3 seeds.")

    wm0, pf0 = res0["model"]
    kg0, tg0, G0, WT0 = surf_g_and_cal(wm0, pf0, *res0["ext_dom"], nk=50, nt=25)
    wmF, pfF = res["model"]
    kgF, tgF, GF, WTF = surf_g_and_cal(wmF, pfF, *res["ext_dom"], nk=50, nt=25)

    fig = make_subplots(rows=2, cols=2, subplot_titles=(
        "g with penalties (lam=10)", "g without penalties (lam=0) — red < 0",
        "d_tau w with penalties (lam=10)", "d_tau w without penalties (lam=0) — red < 0"))
    for (Z, r_, c_, show) in [(GF, 1, 1, True), (G0, 1, 2, False),
                              (WTF, 2, 1, False), (WT0, 2, 2, False)]:
        fig.add_trace(go.Heatmap(z=Z, x=kgF, y=tgF, zmid=0, colorscale="RdBu",
                                 showscale=show, colorbar=dict(title="val")), r_, c_)
        fig.add_trace(viol_overlay(Z, kgF, tgF), r_, c_)
    fig.update_xaxes(title_text="k")
    fig.update_yaxes(title_text="tau", col=1)
    fig.update_layout(width=1000, height=780, title=f"Real-data penalty ablation — {d}")
    fig.show()



[lambda=10 (full)] in 1.562 | holdout 1.525 vp
                   min g +0.1339 | bfly viol   0.0% | pen_bfly nodes 0.00e+00 audit 0.00e+00
                   min d_tau w +0.0039 | cal viol   0.0% | pen_cal  nodes 0.00e+00 audit 0.00e+00
[lambda=0  (none)] in 1.713 | holdout 1.649 vp
                   min g +0.1025 | bfly viol   0.0% | pen_bfly nodes 0.00e+00 audit 0.00e+00
                   min d_tau w -0.2457 | cal viol  19.1% | pen_cal  nodes 1.20e-03 audit 1.36e-03

[WARNING] lambda=0 fits WORSE in-sample (1.713 vs 1.562 vp). Impossible at the optimum: this is an OPTIMISATION
          artefact (equal budget; penalties precondition the descent). Do NOT
          report it as 'the penalties improve accuracy'. Set RUN_LAM0_CHECK=True
          to verify with 3x epochs and 3 seeds.


In [75]:
# --- optional verification of the lambda=0 in-sample paradox (3x epochs, 3 seeds) ---
if RUN_LAM0_CHECK and real_rows and last_day is not None:
    d, res = last_day
    print(f"lambda=0 long-run check on {d} (reference lambda=10 in-sample: "
          f"{res['rmse_in']*100:.3f} vp)")
    for off in (0, 1, 2):
        r0 = run_real_day(df.filter(plr.col("date") == d), d,
                          seed=stable_seed(d) + off, lam_b=0.0, lam_c=0.0,
                          epochs=3 * EPOCHS_REAL)
        print(f"  seed+{off}: in {r0['rmse_in']*100:.3f} vp | "
              f"cal viol {r0['cal_viol_pct_ext']:.1f}% | {r0['runtime_s']:.0f}s")
    print("If the long runs close the gap: pure budget artefact -> say so in one sentence.\n"
          "If the gap persists: a genuine conditioning effect -> interesting, but demonstrate it\n"
          "(loss traces, several seeds) before claiming it.")



## 6. Summary

**Implemented (faithful to the papers):**
- **Ackerer et al. (2020)**: total variance = SSVI prior × positive neural corrector (≈1 at init),
  rescaled inputs, tanh ($C^\infty$) activations, cube-root-dense collocation extending beyond the
  quotes ($\mathcal I_{C45}$), far-wing linearity penalty ($\mathcal I_{C6}$), λ-sweep $\{0,1,10\}$.
- **DCNN (Hoshisashi et al., 2024)**: exact first/second derivatives by autodiff, validated in
  **both** $k$ (against closed-form SVI, ~1e-16) and $\tau$ (against central differences, ~1e-11);
  soft butterfly + calendar penalties on meshes distinct from the quotes.
- **Thesis-wide symmetry**: the fit term carries the same IV-target weights as NB02's SVI/SSVI
  calibrators. Hold-out uses the NB02 protocol (per-`exdate` 20%, `crc32(date)` seed).

**The methodological results of this notebook:**

> **1. Soft no-arbitrage constraints are enforced only where they are sampled, and a collocation
> grid designed for one constraint can be structurally blind to another** (v3: `pen_cal = 0` and a
> 28% calendar-violation rate, simultaneously and truthfully — the exponential τ-grid had a 0.41y
> hole exactly where the stressor forced the crossing). The fix is **constraint-specific
> collocation**: butterfly keeps its ATM-dense hybrid grid; calendar gets a dedicated
> uniform-dense τ grid, affordable because $\partial_\tau w$ costs one first derivative. With it,
> the penalty genuinely arbitrates: 38.1% → 1.1% violations, fitted crossing +0.029 → −0.000, at a
> measured price of ~1.9 vp on the deflated slice — the Ackerer trade-off made visible. Detection
> relied on (i) an audit grid **independent of and finer than** every collocation grid, and
> (ii) **separate** butterfly/calendar violation rates; both are now first-class outputs
> (`pen_*` nodes-vs-audit pairs, `blind_*` flags).
>
> **2. The deep smoother's sparsity robustness is inherited from its prior's calibration
> robustness.** Removing the sparsity-study data leak exposed that a *mis-calibrated* prior is
> worse than a *flat* one for a multiplicative corrector (14.2 vp vs 1.7 vp at 50% subsampling —
> caused by a clamping ATM interpolation). The prior fit is now robust (nearest-ATM proxy, joint
> 5-parameter refinement), the **prior-only RMSE** is reported per run to separate prior failures
> from corrector failures, and the study is **multi-seed**: single subsample draws are lotteries.
> The architecture moves the failure point; it does not remove it.

**Other evidence:**
- **Equal-epoch ablations**: the prior carries accuracy. On clean data the penalty gradients are
  *exactly zero* — the two `no_prior` rows are identical to the last digit and their traces
  coincide on the plot. The prior does the protective work; the penalties are dormant.
- **Calendar stress**: crossing guaranteed by construction; a two-condition **fit-validity gate**
  (deflated-slice RMSE *and* fitted crossing) verifies the stress is actually fitted; a
  **resolution assertion** verifies the calendar grid can see the stressed interval at all.
- **Butterfly**: no synthetic stressor — at small $w$, $g<0$ is slope-driven and needs structure
  finer than a smooth corrector produces at fittable widths. Ablated instead on **real ultra-short
  quotes** (§5b), the dataset's natural stressor.
- **Optimisation hygiene**: Adam state persists across logging chunks (v5 reported all final
  metrics mid-restart-transient); the λ=0-fits-worse-in-sample real-data observation is flagged as
  an optimisation artefact, never as an accuracy benefit of the constraints, with an optional
  long-run verification (`RUN_LAM0_CHECK`).
- **Real SPX**: hold-out vs hold-out against SVI *and* SSVI, maturity buckets, arbitrage audited
  beyond the quotes but capped at $|k|\le 1.5$.

**Outputs:** `deep_smoother_days.parquet` — per-day metrics including
`deep_bfly_viol_pct_ext` / `deep_cal_viol_pct_ext`, the node-vs-audit penalty pairs, the
`deep_blind_*` flags and both `tau_gap_*` diagnostics; input to NB05.

**Next (NB04).** From *one network per day* to *one operator for all days*: Operator Deep
Smoothing (ICLR 2025), trained across days. The operator inherits the collocation problem
wholesale — constraint-specific grids and the node-vs-audit reporting must be carried over.